In this notebook, we unify a few motifes of the previous three notebooks (e.g., including virtual nodes & positional encodings, including substructure counts as node features, including graph-level features such as Morgan fingerprints) into one workflow. Additionally, we also investigate the effect of having a feed-forward neural network at the end of each GNN layer. Luo et al. recently showed that this has the potential to significantly improve GNNs ([Luo et al.: Unlocking the Potential of Classic GNNs for Graph-level Tasks: Simple Architectures Meet Excellence. **2025**](https://arxiv.org/pdf/2502.09263v1)).

We use a ZINC-12k-**like** dataset as our sandbox (because the [ZINC-12k in Pytorch geometric](https://pytorch-geometric.readthedocs.io/en/2.5.2/generated/torch_geometric.datasets.ZINC.html) does not come with the original SMILES strings). For this, we download the full ZINC-250k (ZINC-full) and at each repeat draw randomly 10.000, 1.000 and 1.000 compounds for train, validation, and test dataset respectively.

In [1]:
!pip install numpy==1.23.5

In [2]:
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


In [3]:
!conda install -q -y -c conda-forge rdkit

Channels:
 - conda-forge
Platform: linux-64
Solving environment: ...working... done

# All requested packages already installed.



In [4]:
!pip install pytorch_lightning
!pip install torch_geometric

  Using cached pytorch_lightning-2.5.0.post0-py3-none-any.whl.metadata (21 kB)
  Using cached torch-2.6.0-cp311-cp311-manylinux1_x86_64.whl.metadata (28 kB)
  Using cached PyYAML-6.0.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.1 kB)
  Using cached fsspec-2025.3.0-py3-none-any.whl.metadata (11 kB)
  Using cached torchmetrics-1.6.2-py3-none-any.whl.metadata (20 kB)
  Using cached lightning_utilities-0.14.0-py3-none-any.whl.metadata (5.6 kB)
  Using cached aiohttp-3.11.13-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.7 kB)
  Using cached filelock-3.17.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda

In [5]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
import matplotlib.pyplot as plt
import random
from pathlib import Path
from copy import deepcopy
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from scipy.sparse import coo_matrix
from torch_geometric.data import Data
from torch_geometric.nn.conv import GCNConv
import torch_geometric.nn as nn_pyg
from torch_geometric.nn.aggr import (
    MultiAggregation, MeanAggregation, StdAggregation, SoftmaxAggregation)
from torch_geometric.loader import DataLoader as DataLoader_pyg
import torch_geometric.transforms as T
import pytorch_lightning as pl
from pytorch_lightning.callbacks import (
    ModelCheckpoint, EarlyStopping, LearningRateMonitor)
from pytorch_lightning.loggers import CSVLogger
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
import logging
import warnings
warnings.filterwarnings("ignore", ".*does not have many workers.*")
warnings.filterwarnings("ignore", ".*The number of training batches*")
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)
plt.rcParams["figure.dpi"] = 300
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

n_train = 10000
n_val = 1000
n_test = 1000

In [7]:
class MoleculeProcessor:
    def __init__(self, virtual_node, pe_n, ring_sizes, charge, n_fp,
                 atom_types, bond_types, use_edge_attr):
        if atom_types is None:
            atom_types = []
        self.dic_atoms = {symbol: idx for idx, symbol in enumerate(atom_types)}
        if bond_types is None:
            bond_types = []
        self.dic_bonds = {symbol: idx for idx, symbol in enumerate(bond_types)}
        self.use_edge_attr = use_edge_attr
        self.ring_sizes = ring_sizes
        self.n_fp = n_fp
        if self.n_fp > 0:
            self.mfp_generator = rdFingerprintGenerator.GetMorganGenerator(
                radius=2, fpSize=n_fp)
        self.charge = charge

        transforms = []
        if virtual_node:
            transforms.append(T.VirtualNode())
        if pe_n > 0:
            transforms.append(T.AddLaplacianEigenvectorPE(k=pe_n, attr_name="pe"))
        self.transforms = T.Compose(transforms)

    @staticmethod
    def smile_to_mol(smile):
        return Chem.MolFromSmiles(smile)

    def mol_to_mfp(self, mol):
        mfp = np.zeros((1, self.n_fp), dtype=np.float32)
        mfp_ = self.mfp_generator.GetFingerprint(mol)
        DataStructs.ConvertToNumpyArray(mfp_, mfp[0, :])
        return mfp

    def smile_to_data(self, smile, y=None, ex_feats=None):
        mol = MoleculeProcessor.smile_to_mol(smile)
        edge_idx, _, n_edges = self.get_edge_idx(mol)
        x, node_feats = self.get_x(mol)

        edge_attr = None
        if self.use_edge_attr:
            edge_attr = self.get_edge_attr(mol, edge_idx, n_edges)

        fp = None
        if self.n_fp > 0:
            fp = torch.tensor(self.mol_to_mfp(mol))

        data = Data(
            x=x, edge_index=edge_idx, edge_attr=edge_attr,
            node_feats=node_feats, fp=fp, y=y, ex_feats=ex_feats
            )
        return self.transforms(data)

    def smiles_to_datalist(self, smiles, y=None):
        data = []
        for idx, smile in enumerate(smiles):
            if y is None:
                data.append(self.smile_to_data(smile, None))
            else:
                data.append(
                    self.smile_to_data(smile, torch.tensor([[y[idx]]]).float()))
        return data

    def get_x(self, mol):
        x = np.array([[self.dic_atoms[atom.GetSymbol()] for atom in mol.GetAtoms()]])

        if len(self.ring_sizes) > 0 and not self.charge:
            node_feats = self.get_ring_feats(mol)
        elif len(self.ring_sizes) == 0 and self.charge:
            node_feats = self.get_charge(mol)
        elif len(self.ring_sizes) > 0 and self.charge:
            node_feats = torch.cat(
                [self.get_ring_feats(mol), self.get_charge(mol)], axis=-1)
        else:
            node_feats = None
        return torch.tensor(x.T).type(torch.int32), node_feats

    def get_ring_feats(self, mol):
        n_atoms = len(mol.GetAtoms())
        x = np.zeros((n_atoms, len(self.ring_sizes)))
        rings = mol.GetRingInfo().AtomRings()

        for idx0, ring_size in enumerate(self.ring_sizes):
            rings_cur = [x for x in rings if len(x) == ring_size]
            idx1, counts = np.unique(rings_cur, return_counts=True)
            if idx1.shape[0] > 0:
                x[idx1, idx0] = counts / 2 # / 2 for normalization. atoms can typically be expected to not be part of more than 2 ring systems at once
        return torch.tensor(x).type(torch.float32)

    def get_charge(self, mol):
        node_feats = torch.tensor([[atom.GetFormalCharge()] for atom in mol.GetAtoms()])
        node_feats = (node_feats + 1) / 2 # for normalization. Only formal charge from -1 to +1 are expected in this dataset
        return node_feats

    def get_edge_idx(self, mol):
        adj = coo_matrix(Chem.GetAdjacencyMatrix(mol))
        edge_idx = torch.stack(
            [torch.from_numpy(adj.row).to(torch.long),
             torch.from_numpy(adj.col).to(torch.long)],
            dim=0
            )
        n_nodes = adj.shape[0]
        n_edges = int(np.sum(adj))
        return edge_idx, n_nodes, n_edges

    def get_edge_attr(self, mol, edge_idx, n_edges):
        edge_attr = np.zeros((n_edges,), dtype=np.float32)
        for count, (idx1, idx2) in enumerate(zip(edge_idx[0], edge_idx[1])):
            bond_type = str((mol).GetBondBetweenAtoms(
                int(idx1), int(idx2)).GetBondType())
            edge_attr[count] = self.dic_bonds[bond_type] / (len(self.dic_bonds) - 1)

        edge_attr = torch.from_numpy(edge_attr).type(torch.float32)
        return edge_attr

    def _atom_type(self, smile, idx, check_validity):
        try:
            mol = MoleculeProcessor.smile_to_mol(smile)
            atom_types = [str(atom.GetSymbol()) for atom in mol.GetAtoms()]
            if check_validity:
                invalid = [
                    x for x in atom_types if x not in self.dic_atom.keys()]
                if len(invalid) > 0:
                    raise Exception(f"Atoms {invalid} not allowed")
            return atom_types, []
        except:
            return [], [idx]

    def get_atom_types(self, smiles, check_validity=False):
        atom_types = set()
        idx_exception = []
        for idx, smile in enumerate(smiles):
            atom_types_cur, idx_exception_cur = self._atom_type(
                smile, idx, check_validity)
            atom_types.update(atom_types_cur)
            idx_exception.extend(idx_exception_cur)
        return sorted(atom_types), idx_exception

    def get_bond_types(self, smiles):
        bond_types = set()
        idx_exception = []
        for idx, smile in enumerate(smiles):
            try:
                mol = MoleculeProcessor.smile_to_mol(smile)
                bond_types_cur = [
                    str(bond.GetBondType()) for bond in mol.GetBonds()]
                bond_types.update(bond_types_cur)
            except:
                idx_exception.append(idx)
        return sorted(bond_types), idx_exception


# https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/zinc15_250K_2D.tar.gz
df = pd.read_csv("./drive/MyDrive/zinc15_250K_2D.csv")

processor_ = MoleculeProcessor(False, 0, [], False, 0, None, None, False)
atom_types, _ = processor_.get_atom_types(df["smiles"].tolist())
bond_types, _ = processor_.get_bond_types(df["smiles"].tolist())

In [12]:
enable_progress_bar = False

# 0
kwargs_processor0 = {
    "virtual_node": True,
    "pe_n": 4,
    "ring_sizes": [3, 4, 5, 6, 7, 8],
    "charge": False,
    "n_fp": 256,
    "atom_types": atom_types,
    "bond_types": bond_types,
    "use_edge_attr": True,
    }

kwargs_module0 = {
    "depth": 16,
    "graph_embeddings": 128,
    "embeddings_n": len(atom_types),
    "embeddings_out": int(len(atom_types)**(3/5)),
    "pe_n": kwargs_processor0["pe_n"],
    "ring_sizes": kwargs_processor0["ring_sizes"],
    "charge": kwargs_processor0["charge"],
    "residual": True,
    "norm": True,
    "gcn_dropout": 0.02,
    "mlp": True,
    "n_fp": kwargs_processor0["n_fp"],
    "n_ex_feats": 0,
    "global_dropout": 0.3,
    "mlp_factors": [3, 6, 3],
    "n_feat_out": 1,
    "batch_size": 256,
    "criterion": nn.L1Loss,
    "lr_init": 1e-3,
    "T_0": 30,
    "T_mult": 1,
    "eta_min": 1e-7,
    }

kwargs_train_extra = {
    "patience_es": 61,
    "epochs": 300,
    "y_lim": [0.0, 0.4]
    }

# 1
kwargs_processor1 = deepcopy(kwargs_processor0)
kwargs_module1 = deepcopy(kwargs_module0)
kwargs_module1["mlp"] = False

# 2
kwargs_processor2 = deepcopy(kwargs_processor0)
kwargs_module2 = deepcopy(kwargs_module0)
kwargs_processor2["virtual_node"] = False

# 3
kwargs_processor3 = deepcopy(kwargs_processor0)
kwargs_module3 = deepcopy(kwargs_module0)
kwargs_processor3["pe_n"] = 0
kwargs_module3["pe_n"] = kwargs_processor3["pe_n"]

# 4
kwargs_processor4 = deepcopy(kwargs_processor0)
kwargs_module4 = deepcopy(kwargs_module0)
kwargs_module4["gcn_dropout"] = 0.0

# 5
kwargs_processor5 = deepcopy(kwargs_processor0)
kwargs_module5 = deepcopy(kwargs_module0)
kwargs_processor5["n_fp"] = 0
kwargs_module5["n_fp"] = kwargs_processor5["n_fp"]

In [9]:
def split_dataset(df, n_train, n_val, n_test):
    df_ = deepcopy(df)
    df_train = df_.sample(n=n_train)
    df_ = df_[~df_.index.isin(df_train.index)].reset_index(drop=True)
    df_train = df_train.reset_index(drop=True)

    df_val = df_.sample(n=n_val)
    df_test = df_[~df_.index.isin(df_val.index)].reset_index(drop=True)
    df_val = df_val.reset_index(drop=True)

    # df_test = df_test.sample(n=n_test).reset_index(drop=True)
    return df_train, df_val, df_test


def create_dataloaders(df_train, df_val, df_test, kwargs_processor, kwargs_module):
    processor = MoleculeProcessor(**kwargs_processor)
    ds_train = processor.smiles_to_datalist(
        df_train["smiles"].tolist(), df_train["logp"].values)
    ds_val = processor.smiles_to_datalist(
        df_val["smiles"].tolist(), df_val["logp"].values)
    ds_test = processor.smiles_to_datalist(
        df_test["smiles"].tolist(), df_test["logp"].values)

    loader_train = DataLoader_pyg(
        ds_train, batch_size=kwargs_module["batch_size"], shuffle=True)
    loader_val = DataLoader_pyg(
        ds_val, batch_size=kwargs_module["batch_size"], shuffle=False)
    loader_test = DataLoader_pyg(
        ds_test, batch_size=kwargs_module["batch_size"], shuffle=False)
    return loader_train, loader_val, loader_test

In [10]:
class MLP(nn.Module):
    def __init__(self, neurons, dropout=0.0, activation=nn.ReLU):
        super().__init__()
        modules = []
        for i in range(len(neurons) - 1):
            modules.append(nn.Linear(neurons[i], neurons[i+1]))
            if i != len(neurons) - 2:
                modules.append(activation())
                modules.append(nn.Dropout(dropout))
        self.mlp = nn.Sequential(*modules)
        self.apply(self._initialize)

    def forward(self, x):
        return self.mlp(x)

    def _initialize(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)

class GCNLayer(nn.Module):
    def __init__(self, in_channels, out_channels, residual, norm, dropout, mlp,
                 activation=nn.ReLU):
        super().__init__()
        self.gcn = GCNConv(in_channels, out_channels, improved=True)

        self.act = activation()
        self.residual = residual
        self.norm = norm
        self.mlp = mlp
        if self.norm:
            self.norm0 = nn.BatchNorm1d(out_channels)
            if self.mlp:
                self.norm1 = nn.BatchNorm1d(out_channels)
                self.norm2 = nn.BatchNorm1d(out_channels)

        if self.mlp:
            self.mlp0 = MLP(
                [out_channels, out_channels*2, out_channels], dropout)
        self.apply(self._initialize)

    def forward(self, x, edge_index, edge_weight, pe):
        x_in = x

        if pe is not None:
            x = torch.cat([x, pe], axis=-1)
        x = self.gcn(x, edge_index, edge_weight)
        if self.norm:
            x = self.norm0(x)
        x = self.act(x)
        if self.residual:
            x = x + x_in
        if self.mlp:
            if self.norm:
                x = self.norm1(x)
            x = self.mlp0(x)
            if self.norm:
                x = self.norm2(x)
        return x

    def _initialize(self, module):
        if isinstance(module, (nn.Linear, nn_pyg.dense.linear.Linear)):
            torch.nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)


class GNNEncoder(nn.Module):
    def __init__(
            self, depth, graph_embeddings, embeddings_n, embeddings_out, pe_n,
            ring_sizes, charge, residual, norm, dropout, mlp):
        super().__init__()
        self.embed = nn.Embedding(embeddings_n, embeddings_out)
        self.gnn = nn.ModuleList(
            [GCNLayer(
                in_channels=embeddings_out+pe_n+len(ring_sizes)+charge,
                out_channels=graph_embeddings,
                residual=False,
                norm=norm,
                dropout=dropout,
                mlp=mlp
                )] +
            [GCNLayer(
                in_channels=graph_embeddings+pe_n,
                out_channels=graph_embeddings,
                residual=residual,
                norm=norm,
                dropout=dropout,
                mlp=mlp
                )
             for _ in range(depth - 1)]
            )
        self.agg = MultiAggregation(
            [
                MeanAggregation(),
                StdAggregation(),
                SoftmaxAggregation(t=0.1, learn=True)
                ]
            )

    def forward(self, batch):
        x = self.embed(batch.x[:, 0].type(torch.int32))
        if hasattr(batch, "node_feats"):
            x = torch.cat([x, batch.node_feats], axis=-1)

        for gnn_layer in self.gnn:
            if hasattr(batch, "pe"):
                x = gnn_layer(x, batch.edge_index, batch.edge_attr, batch.pe)
            else:
                x = gnn_layer(x, batch.edge_index, batch.edge_attr, None)

        x = self.agg(x, batch.batch)
        return x


class GNN(nn.Module):
    def __init__(
        self, depth, graph_embeddings, embeddings_n, embeddings_out, pe_n,
        ring_sizes, charge, residual, norm, gcn_dropout, mlp, n_fp, n_ex_feats,
        global_dropout, mlp_factors, n_feat_out):
        super().__init__()
        self.encoder = GNNEncoder(
            depth=depth,
            graph_embeddings=graph_embeddings,
            embeddings_n=embeddings_n,
            embeddings_out=embeddings_out,
            pe_n=pe_n,
            ring_sizes=ring_sizes,
            charge=charge,
            residual=residual,
            norm=norm,
            dropout=gcn_dropout,
            mlp=mlp
            )
        mlp_neurons = [(graph_embeddings * f) + n_fp + n_ex_feats
                       for f in mlp_factors] + [n_feat_out]
        self.global_dropout = global_dropout
        self.mlp = MLP(mlp_neurons, dropout=0.0)

    def forward(self, batch):
        x = self.encoder(batch)

        if hasattr(batch, "fp"):
            x = torch.cat([x, batch.fp], axis=-1)
        if hasattr(batch, "ex_feats"):
            x = torch.cat([x, batch.ex_feats], axis=-1)
        x = F.dropout(x, self.global_dropout, training=self.training)
        x = self.mlp(x)
        return x


class GNNModule(pl.LightningModule):
    def __init__(
            self, depth, graph_embeddings, embeddings_n, embeddings_out, pe_n,
            ring_sizes, charge, residual, norm, gcn_dropout, mlp, n_fp,
            n_ex_feats, global_dropout, mlp_factors, n_feat_out, batch_size,
            criterion, lr_init, T_0, T_mult, eta_min
            ):
        super().__init__()
        self.gnn = GNN(
            depth=depth,
            graph_embeddings=graph_embeddings,
            embeddings_n=embeddings_n,
            embeddings_out=embeddings_out,
            pe_n=pe_n,
            ring_sizes=ring_sizes,
            charge=charge,
            residual=residual,
            norm=norm,
            gcn_dropout=gcn_dropout,
            mlp=mlp,
            n_fp=n_fp,
            n_ex_feats=n_ex_feats,
            global_dropout=global_dropout,
            mlp_factors=mlp_factors,
            n_feat_out=n_feat_out
            )

        self.batch_size = batch_size
        self.criterion = criterion()
        self.lr_init = lr_init
        self.T_0 = T_0
        self.T_mult = T_mult
        self.eta_min = eta_min

    def forward(self, batch):
        return self.gnn(batch)

    def training_step(self, batch, batch_idx):
        y = batch.y
        y_pred = self.forward(batch)

        loss = self.criterion(y_pred, y)
        self.log("train_loss", loss, on_step=False, on_epoch=True,
                 prog_bar=True, batch_size=self.batch_size)
        return loss

    def validation_step(self, batch, batch_idx):
        y = batch.y
        y_pred = self.forward(batch)

        loss = self.criterion(y_pred, y)
        self.log("val_loss", loss, on_step=False, on_epoch=True,
                 prog_bar=True, batch_size=self.batch_size)
        return loss

    @torch.no_grad()
    def predict_step(self, batch, batch_idx):
        self.gnn.eval()
        y_pred = self.forward(batch)
        return y_pred

    def configure_optimizers(self):
        self.opt = torch.optim.Adam(self.parameters(), lr=self.lr_init)
        self.train_scheduler = {
            "scheduler": CosineAnnealingWarmRestarts(
                optimizer=self.opt, T_0=self.T_0, T_mult=self.T_mult,
                eta_min=self.eta_min),
            "interval": "epoch",
            "frequency": 1,
            "monitor": "val_loss"}
        return [self.opt], [self.train_scheduler]



def display_training_progress(trainer, y_lim):
    logs = pd.read_csv(Path(trainer.logger.log_dir + "/metrics.csv"))
    epochs = logs.dropna(subset=["train_loss"])["epoch"]

    fig, ax0 = plt.subplots(figsize=(9, 4))
    l0 = ax0.plot(epochs, logs["train_loss"].dropna(), color="r",
                  label="Training loss")
    l1 = ax0.plot(epochs, logs["val_loss"].dropna(), color="b",
                  label="Validation loss")
    ax0.set_xlabel("Epoch")
    ax0.set_ylabel("Loss")
    ax0.set_xlim(0, len(epochs)-1)

    ax1 = ax0.twinx()
    l2 = ax1.plot(
        epochs, logs["lr-Adam"].dropna(), color="g", linestyle="dashed",
        label="Learning rate")
    ax1.set_ylabel("Learning rate")
    ax1.set_yscale("log")
    ax0.set_ylim(y_lim[0], y_lim[1])
    ax0.legend(l0+l1+l2, [l.get_label() for l in l0+l1+l2], loc="lower center",
               bbox_to_anchor=(0, 1.05, 1, 0.2), ncol=3)
    plt.tight_layout()
    ax0.grid()
    plt.show()
    return fig

In [13]:
def train_model(name, df, n_train, n_val, n_test, kwargs_processor,
                kwargs_module, kwargs_train_extra):
    df_train, df_val, df_test = split_dataset(df, n_train, n_val, n_test)
    loader_train, loader_val, loader_test = create_dataloaders(
        df_train, df_val, df_test, kwargs_processor, kwargs_module)
    m = GNNModule(**kwargs_module)

    cb_a = ModelCheckpoint(monitor="val_loss", mode="min")
    cb_b = EarlyStopping(
        monitor="val_loss", mode="min", patience=kwargs_train_extra["patience_es"])
    cb_c = LearningRateMonitor(logging_interval="epoch")
    logger = CSVLogger(save_dir="logs/", name=name, flush_logs_every_n_steps=1)

    trainer = pl.Trainer(
        max_epochs=kwargs_train_extra["epochs"], check_val_every_n_epoch=1,
        callbacks=[cb_a, cb_b, cb_c], logger=logger, accelerator="gpu",
        enable_progress_bar=enable_progress_bar
        )
    trainer.fit(
        model=m, train_dataloaders=loader_train, val_dataloaders=loader_val)

    # fig = display_training_progress(trainer, kwargs_train_extra["y_lim"])

    checkpoint_file = trainer.checkpoint_callback.best_model_path
    m = GNNModule.load_from_checkpoint(checkpoint_file, **kwargs_module)

    y_test_pred = torch.cat(
        trainer.predict(m, dataloaders=loader_test)).detach().numpy()
    y_test_true = torch.cat([b.y for b in loader_test]).detach().numpy()
    mae = np.mean(np.abs(y_test_pred - y_test_true))
    return mae

def train_model_repeated(
        n_repeats, name, df, n_train, n_val, n_test, kwargs_processor,
        kwargs_module, kwargs_train_extra):
    maes = []
    for i in range(n_repeats):
        maes.append(train_model(
            name + f"_{i}", df, n_train, n_val, n_test, kwargs_processor,
            kwargs_module, kwargs_train_extra))
    # print(maes)
    return np.mean(np.array(maes)), np.std(np.array(maes))


n_reps = 3
result0 = train_model_repeated(
    n_reps, "m0", df, n_train, n_val, n_test, kwargs_processor0, kwargs_module0,
    kwargs_train_extra)
result1 = train_model_repeated(
    n_reps, "m1", df, n_train, n_val, n_test, kwargs_processor1, kwargs_module1,
    kwargs_train_extra)
result2 = train_model_repeated(
    n_reps, "m2", df, n_train, n_val, n_test, kwargs_processor2, kwargs_module2,
    kwargs_train_extra)
result3 = train_model_repeated(
    n_reps, "m3", df, n_train, n_val, n_test, kwargs_processor3, kwargs_module3,
    kwargs_train_extra)
result4 = train_model_repeated(
    n_reps, "m4", df, n_train, n_val, n_test, kwargs_processor4, kwargs_module4,
    kwargs_train_extra)
result5 = train_model_repeated(
    n_reps, "m5", df, n_train, n_val, n_test, kwargs_processor5, kwargs_module5,
    kwargs_train_extra)

In [14]:
columns = [
    "virtual_node", "pe_n", "ring_sizes", "charge", "n_fp", "use_edge_attr",
    "gcn_dropout", "mlp", "MAE (mean)", "MAE (std)"]

def format_kwargs(columns, kwargs_processor, kwargs_module, result):
    out = []
    for col in columns:
        try:
            out.append(kwargs_processor[col])
        except:
            try:
                out.append(kwargs_module[col])
            except:
                None
    out.extend(result)
    return out

configs = [
    (kwargs_processor0, kwargs_module0, result0),
    (kwargs_processor1, kwargs_module1, result1),
    (kwargs_processor2, kwargs_module2, result2),
    (kwargs_processor3, kwargs_module3, result3),
    (kwargs_processor4, kwargs_module4, result4),
    (kwargs_processor5, kwargs_module5, result5),
    ]

results = [format_kwargs(columns, kwargs0, kwargs1, result) for (kwargs0, kwargs1, result) in configs]

df = pd.DataFrame.from_records(results, columns=columns)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 1000)
print(df)

   virtual_node  pe_n          ring_sizes  charge  n_fp  use_edge_attr  gcn_dropout    mlp  MAE (mean)  MAE (std)
0          True     4  [3, 4, 5, 6, 7, 8]   False   256           True         0.02   True    0.086648   0.002977
1          True     4  [3, 4, 5, 6, 7, 8]   False   256           True         0.02  False    0.134306   0.003720
2         False     4  [3, 4, 5, 6, 7, 8]   False   256           True         0.02   True    0.095412   0.007034
3          True     0  [3, 4, 5, 6, 7, 8]   False   256           True         0.02   True    0.095067   0.016048
4          True     4  [3, 4, 5, 6, 7, 8]   False   256           True         0.00   True    0.089098   0.005054
5          True     4  [3, 4, 5, 6, 7, 8]   False     0           True         0.02   True    0.063844   0.016482


As outlined by Luo et al. ([Luo et al.: Unlocking the Potential of Classic GNNs for Graph-level Tasks: Simple Architectures Meet Excellence. **2025**](https://arxiv.org/pdf/2502.09263v1)) including a feed-forward network a the end of each GNN layer truely massively improves results. Results are in the ballpark of recent state-of-the-art architectures (e.g., [TIGT](https://arxiv.org/pdf/2402.02005) achieves a MAE of 0.057 on ZINC-12k, [GRIT](https://arxiv.org/pdf/2305.17589) a MAE of 0.059.)